In [0]:
# Databricks notebook source

from __future__ import annotations

import uuid

# ============================================================
# Widgets
# ============================================================
dbutils.widgets.text("catalog", "phc_txdot")
catalog = dbutils.widgets.get("catalog").strip()

# ============================================================
# Constants
# ============================================================
SCRIPT_NAME = "410_build_gold_vw_dim_project.py"
RUN_ID = str(uuid.uuid4())

BASE_TABLE = f"{catalog}.silver.closed_bids_base_clean"
VENDOR_LATEST_TABLE = f"{catalog}.silver.closed_bids_vendor_item_latest"
DIM_PROJECT_TABLE = f"{catalog}.silver.dim_project"
DIM_PROJECT_CLASSIFICATION_TABLE = f"{catalog}.silver.dim_project_classification"
DIM_COUNTY_TABLE = f"{catalog}.silver.dim_county"
TARGET_VIEW = f"{catalog}.gold.vw_dim_project"
RUN_LOG_TABLE = f"{catalog}.staging.pipeline_run_log"

# ============================================================
# Helpers
# ============================================================
def sql_escape(value: str) -> str:
    return value.replace("'", "''")


def log_run(status: str, row_count: int | None, message: str) -> None:
    row_count_sql = "NULL" if row_count is None else str(row_count)
    spark.sql(f"""
        INSERT INTO {RUN_LOG_TABLE}
        VALUES (
              '{RUN_ID}'
            , '{SCRIPT_NAME}'
            , '{sql_escape(status)}'
            , {row_count_sql}
            , '{sql_escape(message)}'
            , current_timestamp()
        )
    """)

# ============================================================
# Build gold.vw_dim_project
# ============================================================
try:
    spark.sql(f"DROP VIEW IF EXISTS {TARGET_VIEW}")

    spark.sql(f"""
    CREATE VIEW {TARGET_VIEW} AS
    WITH vendor_stats AS (
        SELECT
              project_id
            , COUNT(DISTINCT CASE WHEN row_type = 'VENDOR' THEN vendor_name_standardized END) AS vendor_count_excluding_estimate
            , COUNT(DISTINCT vendor_name_standardized) AS vendor_count_including_estimate
            , COUNT(DISTINCT CASE WHEN row_type = 'VENDOR' THEN business_submission_key END) AS vendor_submission_count
        FROM {BASE_TABLE}
        GROUP BY project_id
    )

    , item_seq_stats AS (
        SELECT
              project_id
            , COUNT(DISTINCT bid_item_sequence_number) AS distinct_vendor_item_sequence_count
        FROM {VENDOR_LATEST_TABLE}
        GROUP BY project_id
    )

    SELECT
          dp.project_key                                                   AS `Project Key`
        , dp.project_id                                                    AS `Project ID`

        , pc.project_classification_key                                    AS `Project Classification Key`
        , dc.county_key                                                    AS `County Key`

        , dp.project_name                                                  AS `Project Name`
        , dp.project_number                                                AS `Project Number`
        , dp.state_project_number                                          AS `State Project Number`
        , dp.control_section_job_csj                                       AS `Control Section Job CSJ`
        , dp.controlling_project_id_ccsj                                   AS `Controlling Project ID CCSJ`
        , dp.project_type                                                  AS `Project Type`
        , dp.project_classification                                        AS `Project Classification`
        , dp.project_actual_let_date                                       AS `Project Actual Let Date`
        , dp.project_estimated_let_date                                    AS `Project Estimated Let Date`
        , dp.project_limits_from                                           AS `Project Limits From`
        , dp.project_limits_to                                             AS `Project Limits To`
        , dp.county                                                        AS `County`
        , dc.county_location                                               AS `Location`
        , dp.district_division                                             AS `District Division`
        , dp.highway                                                       AS `Highway`
        , dp.short_description                                             AS `Short Description`
        , dp.federal_project_number                                        AS `Federal Project Number`

        , COALESCE(vs.vendor_count_including_estimate, 0)                  AS `Vendor Count Including Estimate`
        , COALESCE(vs.vendor_count_excluding_estimate, 0)                  AS `Vendor Count Excluding Estimate`
        , COALESCE(vs.vendor_submission_count, 0)                          AS `Vendor Submission Count`
        , COALESCE(iss.distinct_vendor_item_sequence_count, 0)             AS `Distinct Vendor Item Sequence Count`
        , COALESCE(dp.vendor_latest_item_count, 0)                         AS `Vendor Latest Item Count`
        , COALESCE(dp.vendor_latest_vendor_count, 0)                       AS `Vendor Latest Vendor Count`
        , COALESCE(dp.estimate_current_item_count, 0)                      AS `Estimate Current Item Count`
        , dp.min_engineer_project_total                                    AS `Min Engineer Project Total`
        , dp.max_engineer_project_total                                    AS `Max Engineer Project Total`
        , COALESCE(dp.has_engineer_estimate, FALSE)                        AS `Has Engineer Estimate Rows`
        , COALESCE(dp.has_conflicting_engineer_project_totals, FALSE)      AS `Has Conflicting Engineer Project Totals`

        , dp.actual_let_year                                               AS `Actual Let Year`
        , dp.actual_let_month                                              AS `Actual Let Month`
        , dp.estimated_let_year                                            AS `Estimated Let Year`
        , dp.estimated_let_month                                           AS `Estimated Let Month`
        , dp.estimated_to_actual_let_day_diff                              AS `Estimated To Actual Let Day Diff`

        , dp.source_updated_at                                             AS `Latest Source Updated At`
        , dp.ingested_at_utc                                               AS `Latest Load Timestamp`

    FROM {DIM_PROJECT_TABLE} dp
    LEFT JOIN {DIM_PROJECT_CLASSIFICATION_TABLE} pc
        ON (
            CASE
                WHEN COALESCE(dp.project_classification, '') = '' THEN 'UNKNOWN'
                ELSE dp.project_classification
            END
        ) = pc.project_classification
    LEFT JOIN {DIM_COUNTY_TABLE} dc
        ON (
            CASE
                WHEN COALESCE(dp.county, '') = '' THEN 'UNKNOWN'
                WHEN dp.county = 'De Witt' THEN 'DeWitt'
                ELSE dp.county
            END
        ) = dc.county
    LEFT JOIN vendor_stats vs
        ON dp.project_id = vs.project_id
    LEFT JOIN item_seq_stats iss
        ON dp.project_id = iss.project_id
    """)

    row_count = spark.sql(f"SELECT COUNT(*) AS row_count FROM {TARGET_VIEW}").collect()[0]["row_count"]

    log_run("SUCCESS", row_count, "Built gold.vw_dim_project successfully.")

    print("=" * 70)
    print("Built gold.vw_dim_project")
    print(f"Row count: {row_count:,}")
    print("=" * 70)

except Exception as exc:
    log_run("FAILED", None, str(exc))
    raise